# Run AutoMetrics on Press Release Dataset

In [2]:
import os
import sys
import pandas as pd
import dspy

sys.path.append("methods/autometrics")

from autometrics.autometrics import Autometrics
from autometrics.dataset.Dataset import Dataset
from autometrics.recommend.LLMRec import LLMRec

/Users/spangher/miniconda3/lib/python3.12/site-packages/pyemd/__init__.py:74: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from .emd import emd, emd_with_flow, emd_samples


[Autometrics] No GPU detected - using BM25 + LLMRec pipeline for CPU-optimized performance


In [3]:
df = pd.read_csv("../datasets/press-releases/press_release_modeling_dataset.csv.gz", compression="gzip", usecols=["press_release_id", "text", "judgement"])
df = df.dropna(subset=["press_release_id", "text", "judgement"]).sample(n=300, random_state=42)
df["judgement"] = df["judgement"].astype(int)
df = df.rename(columns={"press_release_id": "id", "text": "output", "judgement": "newsworthiness_score"})
df.head()

,id,output,newsworthiness_score
26031,90518,"Boeing \n +share \nMQ-28 A smart, uncrewed fo...",1
80556,78941,Sign in SUBSCRIBE Help & Support Knowledge Bas...,0
112836,144580,Thomas'(r) Expands Swirl Bread Line Nationwide...,0
4704,18444,\nLOW RATES & FEES \nLow Rates \nBalance Trans...,1
15315,56754,"\nAug. 27, 2021 \nStatement Regarding a Collis...",1


In [4]:
dataset = Dataset(
    dataframe=df,
    target_columns=["newsworthiness_score"],
    ignore_columns=[],
    metric_columns=[],
    name="PressReleaseModeling",
    data_id_column="id",
    input_column="output",
    output_column="output",
    reference_columns=None,
    task_description="Estimate whether each press release example should receive a positive judgement label."
)

In [5]:
import os 
os.environ['OPENAI_API_KEY'] = open('/Users/spangher/.openai-salt-lab-key.txt').read().strip()

In [10]:
use_openai = False
if use_openai: 
    generator_llm = dspy.LM("openai/gpt-5-mini", api_key=os.environ["OPENAI_API_KEY"])
    judge_llm = dspy.LM("openai/gpt-5-mini", api_key=os.environ["OPENAI_API_KEY"])

else:
    generator_llm = dspy.LM(
        "openai/llama-3.3-70b",
        api_base="http://localhost:8000/v1",
        api_key="EMPTY",
    )
    
    judge_llm = dspy.LM(
        "openai/llama-3.3-70b",
        api_base="http://localhost:8000/v1",
        api_key="EMPTY",
    )    

In [11]:
metric_generation_configs = {
    "llm_judge": {"metrics_per_trial": 5},
    "rubric_dspy": {"metrics_per_trial": 5},
}

autometrics = Autometrics(
    metric_generation_configs=metric_generation_configs,
    retriever=LLMRec,
    retriever_kwargs={"index_path": None, "force_reindex": False},
    metric_bank=[],
    allowed_failed_metrics=1
)

In [11]:
results = autometrics.run(
    dataset=dataset,
    target_measure="newsworthiness_score",
    generator_llm=generator_llm,
    judge_llm=judge_llm,
    num_to_retrieve=20,
    num_to_regress=10
)

[Autometrics] Starting pipeline for PressReleaseModeling - newsworthiness_score
[Autometrics] Configuration: retrieve=20, regress=10, regenerate=False

[Autometrics] Step 1: Generating/Loading Metrics
[Autometrics] Loaded 10 existing metrics for llm_judge
[Autometrics] Warning: Failed to import PressReleaseModeling_newsworthiness_score_rubric_dspy_seed42_metric04.py: expected ':' (PressReleaseModeling_newsworthiness_score_rubric_dspy_seed42_metric04.py, line 11)
[Autometrics] Warning: Failed to import PressReleaseModeling_newsworthiness_score_rubric_dspy_seed42_metric01.py: expected ':' (PressReleaseModeling_newsworthiness_score_rubric_dspy_seed42_metric01.py, line 11)
[Autometrics] Warning: Failed to import PressReleaseModeling_newsworthiness_score_rubric_dspy_seed42_metric05.py: expected ':' (PressReleaseModeling_newsworthiness_score_rubric_dspy_seed42_metric05.py, line 11)
[Autometrics] Warning: Failed to import PressReleaseModeling_newsworthiness_score_rubric_dspy_seed42_metric02.p

2026/03/02 23:36:27 WARNING dspy.primitives.module: Calling module.forward(...) on GenerateRubric directly is discouraged. Please use module(...) instead.
2026/03/02 23:36:27 WARNING dspy.primitives.module: Calling module.forward(...) on GenerateRubric directly is discouraged. Please use module(...) instead.
2026/03/02 23:36:27 WARNING dspy.primitives.module: Calling module.forward(...) on GenerateRubric directly is discouraged. Please use module(...) instead.
2026/03/02 23:36:27 WARNING dspy.primitives.module: Calling module.forward(...) on GenerateRubric directly is discouraged. Please use module(...) instead.
2026/03/02 23:36:27 WARNING dspy.primitives.module: Calling module.forward(...) on GenerateRubric directly is discouraged. Please use module(...) instead.


DEBUG: Generated 5 axes
DEBUG: Starting rubric generation with ThreadPoolExecutor...
DEBUG: Submitting rubric generation task 1/5 for 'Specificity of Information'
DEBUG: Generating rubric for 'Specificity of Information' in thread
  Trying rubric generation with full examples...
DEBUG: Submitting rubric generation task 2/5 for 'Presence of Financial Data'
DEBUG: Generating rubric for 'Presence of Financial Data' in thread
  Trying rubric generation with full examples...
DEBUG: Submitting rubric generation task 3/5 for 'Tone of the Text'
DEBUG: Generating rubric for 'Tone of the Text' in thread
  Trying rubric generation with full examples...
DEBUG: Submitting rubric generation task 4/5 for 'Length of the Text'
DEBUG: Generating rubric for 'Length of the Text' in thread
  Trying rubric generation with full examples...
DEBUG: Submitting rubric generation task 5/5 for 'Use of Technical Jargon'
DEBUG: Generating rubric for 'Use of Technical Jargon' in thread
  Trying rubric generation with


KeyboardInterrupt



KeyboardInterrupt: 

  Success with full examples (seed=42, temp=0.004200000000000001)
  Success with full examples (seed=42, temp=0.004200000000000001)
  Success with full examples (seed=42, temp=0.004200000000000001)  Non-context error in rubric generation: Socket operation on non-socket
  Success with full examples (seed=42, temp=0.004200000000000001)


In [9]:
{
    "top_metrics": [m.get_name() for m in results["top_metrics"]],
    "regression_metric": None if results["regression_metric"] is None else results["regression_metric"].get_name(),
    "report_card_path": results["report_card_path"]
}

{'top_metrics': ['Timeliness_and_Novelty_gpt-5-mini',
  'Public_Impact_and_Audience_Reach_gpt-5-mini',
  'Originality_and_Source_Attribution_gpt-5-mini'],
 'regression_metric': 'Autometrics_Regression_newsworthiness_score',
 'report_card_path': 'artifacts/report_PressReleaseModeling_newsworthiness_score_42.html'}